# CITS3011 Lab 1: Maze Agent

**Student workbook**

This formative, unassessed notebook is intended for students to work on before the Thursday lecture-tutorial.

Work from top to bottom. Run each code cell with **Shift+Enter**. In Google
Colab, save your own copy before editing if you want to retain your work. You
can also download and work it as a local notebook.


## Learning objectives

This Week 1 lab is a teaser: it lets you experience the challenge of
designing an agent that must act without seeing its whole environment.
You are not expected to know a search algorithm yet. Experiment with
your own ideas, observe where they succeed or fail, and use those
observations to improve the agent.

By the end of this lab, you should be able to:

- describe an agent in terms of percepts, actions, internal state, and goals;
- use the outcome of an action to update an internal model;
- implement exploration in unknown environments; and
- distinguish online exploration from an offline shortest path.

The agent begins at `(10, 10)` and must reach `(0, 0)` within 300
actions. It observes only its current coordinates. An attempted move
into a wall or beyond the grid leaves it in the same location.

This is a formative activity and does not contribute to the unit mark.


## Task

Attempt to complete `MazeAgent.reset` and `MazeAgent.get_next_move`.
There is no single required approach: begin with your own strategy and
refine it using the checks and visualisation later in the notebook.

`get_next_move(x, y)` must return one of `"L"`, `"R"`, `"U"`, or
`"D"`. Your agent should build a model from its experience rather than
reading the maze map directly. A possible strategy is depth-first exploration:

1. remember visited and blocked coordinates;
2. try an unvisited neighbouring coordinate;
3. infer a wall when an attempted move changes nothing; and
4. backtrack when no unexplored neighbour remains.

Before coding, decide what states to maintain.


In [ ]:
ACTIONS = {
    "L": (-1, 0),
    "D": (0, -1),
    "R": (1, 0),
    "U": (0, 1),
}
REVERSE = {"L": "R", "R": "L", "U": "D", "D": "U"}

class MazeAgent:
    def reset(self):
        # TODO: initialise the internal state used in a new maze.
        pass

    def get_next_move(self, x, y):
        # TODO: update the model from the previous action outcome, then
        # choose and return one of "L", "R", "U", or "D".
        pass


## Environment and checks

The following cells implement the environment, not the agent. The
maze uses `(x, y)` coordinates: `x` increases from left to right and
`y` increases from bottom to top. In this 11×11 maze, `(0, 0)` is the
bottom-left goal, `(10, 10)` is the top-right starting position,
`(0, 10)` is the top-left corner, and `(10, 0)` is the bottom-right
corner. Therefore, when the maze is printed row by row, the first
row represents `y = 10` and the last row represents `y = 0`.

The agent is not given the maze array.


In [ ]:
import matplotlib.pyplot as plt
import random
from collections import deque

GRID_SIZE = 11
MAX_MOVES = 300
KNOWN_MAZES = [
"""#...####...
##.#...#.#.
#..#.#.#.#.
##.#.#.#.#.
#..#.#.#.#.
##.#.#.#.#.
...#.#.#.#.
.#...#...#.
.#########.
.##########
...........""",
"""...........
.#########.
.#########.
.##......#.
.##.####.#.
.##.#..#.#.
.##.#..#.#.
.##.#..#.#.
.##.#....#.
.##.######.
.##........""",
""".#...#.....
...#.......
...#..#....
...#.......
...#.......
...........
...........
......#....
......#....
...........
.#....#....""",
""".#.##......
.#####.##.#
.#####.##.#
.#..##.###.
.........#.
.#........#
.....#.#...
####.##.#..
##.....##..
#..#.#.#...
...##...#..""",
]

def parse_maze(text):
    rows = text.splitlines()
    return [[rows[GRID_SIZE - 1 - y][x] == "." for x in range(GRID_SIZE)]
            for y in range(GRID_SIZE)]

def generate_maze(seed):
    # Generate a repeatable, previously unseen perfect maze.
    rng = random.Random(seed)
    maze = [[False for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
    stack = [(0, 0)]
    maze[0][0] = True
    while stack:
        x, y = stack[-1]
        candidates = []
        for dx, dy in [(2, 0), (-2, 0), (0, 2), (0, -2)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE and not maze[ny][nx]:
                candidates.append((nx, ny, dx, dy))
        if not candidates:
            stack.pop()
            continue
        nx, ny, dx, dy = rng.choice(candidates)
        maze[y + dy // 2][x + dx // 2] = True
        maze[ny][nx] = True
        stack.append((nx, ny))
    return maze

def run_maze(agent, maze, max_moves=MAX_MOVES):
    x, y = GRID_SIZE - 1, GRID_SIZE - 1
    trace = [(x, y)]
    agent.reset()
    error = None
    for _ in range(max_moves):
        if (x, y) == (0, 0):
            break
        try:
            move = agent.get_next_move(x, y)
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"
            break
        if move not in ACTIONS:
            error = f"Invalid action {move!r}"
            break
        dx, dy = ACTIONS[move]
        nx, ny = x + dx, y + dy
        if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE and maze[ny][nx]:
            x, y = nx, ny
        trace.append((x, y))
    return {"success": (x, y) == (0, 0), "moves": len(trace) - 1,
            "trace": trace, "error": error}

def offline_shortest_path(maze):
    # This is a privileged comparison baseline for analysis. The
    # MazeAgent itself never receives or searches the maze array.
    start, goal = (10, 10), (0, 0)
    queue = deque([start])
    parent = {start: None}
    while queue:
        x, y = queue.popleft()
        if (x, y) == goal:
            path = []
            current = goal
            while current is not None:
                path.append(current)
                current = parent[current]
            return path[::-1]
        for dx, dy in ACTIONS.values():
            nxt = (x + dx, y + dy)
            if (0 <= nxt[0] < GRID_SIZE and 0 <= nxt[1] < GRID_SIZE
                    and maze[nxt[1]][nxt[0]] and nxt not in parent):
                parent[nxt] = (x, y)
                queue.append(nxt)
    return None

def shortest_path_length(maze):
    path = offline_shortest_path(maze)
    return None if path is None else len(path) - 1

def show_run(maze, result, title="Maze-agent run", offline_path=None):
    import numpy as np
    image = np.array(maze, dtype=int)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image, origin="lower", cmap="Greys_r", vmin=0, vmax=1)
    xs, ys = zip(*result["trace"])
    ax.plot(xs, ys, color="#0072B2", linewidth=2, alpha=.8, label="agent trace")
    if offline_path:
        offline_x, offline_y = zip(*offline_path)
        ax.plot(offline_x, offline_y, color="black", linestyle="--",
                linewidth=2, alpha=.8, label="offline shortest path")
    ax.scatter([10], [10], c="#E69F00", s=90, edgecolors="black",
               zorder=3, label="start")
    ax.scatter([0], [0], c="#009E73", s=90, edgecolors="black",
               zorder=3, label="goal")
    ax.set(xticks=range(11), yticks=range(11), xlabel="x", ylabel="y",
           title=title)
    ax.grid(color="lightgrey", linewidth=.4)
    ax.legend(loc="upper left")
    plt.show()


In [ ]:
test_mazes = [parse_maze(text) for text in KNOWN_MAZES]
test_mazes += [generate_maze(3011), generate_maze(2026)]
results = []
for index, maze in enumerate(test_mazes, start=1):
    result = run_maze(MazeAgent(), maze)
    results.append(result)
    status = "PASS" if result["success"] else "NOT YET"
    detail = result["error"] or f'{result["moves"]} moves'
    print(f"Maze {index}: {status} — {detail}")


## Visual analysis

Choose one of the four supplied mazes, or generate a previously
unseen maze by changing `generated_seed`. In Colab, the next cell
presents these as form controls; in Jupyter, edit the two values
directly and rerun the cell.

Inspect the complete action trace. Repeated coordinates indicate
unsuccessful attempts to enter walls. Compare the number of online
actions with the shortest path available to an agent that already
knows the whole maze.


In [ ]:
maze_choice = "Generated maze" # @param ["Known maze 1", "Known maze 2", "Known maze 3", "Known maze 4", "Generated maze"]
generated_seed = 3011 # @param {type:"integer"}

if maze_choice == "Generated maze":
    maze = generate_maze(generated_seed)
    maze_label = f"Generated maze (seed {generated_seed})"
else:
    maze_index = int(maze_choice.rsplit(" ", 1)[-1]) - 1
    maze = parse_maze(KNOWN_MAZES[maze_index])
    maze_label = maze_choice

result = run_maze(MazeAgent(), maze)
if result["success"]:
    offline_path = offline_shortest_path(maze)
    show_run(maze, result, title=f"{maze_label}: online versus offline",
             offline_path=offline_path)
    print("Online actions:", result["moves"])
    print("Offline shortest-path actions:", len(offline_path) - 1)
    print("Unsuccessful move attempts:", sum(a == b for a, b in zip(result["trace"], result["trace"][1:])))
else:
    print("Complete the agent first:", result["error"] or "goal not reached")


## Further Questions

1. Which percept tells the agent that the target cell is blocked?
2. What makes it challenging for an agent to act effectively in an
   unknown environment when it can observe only its current location
   and the outcome of its previous action?
3. Why can this agent take more actions than the offline shortest
   path even when its exploration strategy is correct?
4. Change the action ordering in `ACTIONS`. Which mazes become
   faster or slower?
